In [ ]:
# ============================================================
# CELL 1: BELT ENSEMBLE-SIZE ABLATION
# ============================================================

import os
import copy
import json
import random
import numpy as np
import pandas as pd
import torch

from collections import OrderedDict
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# Ensemble sizes to evaluate
# ------------------------------------------------------------

ENSEMBLE_SIZES = [2, 5, 10, 15, 20]

# We train the maximum number only once.
MAX_MEMBERS = max(ENSEMBLE_SIZES)

# Independent seeds for the 20 BELT members
MEMBER_SEEDS = [
    42, 52, 62, 72, 82,
    92, 102, 112, 122, 132,
    142, 152, 162, 172, 182,
    192, 202, 212, 222, 232
]

assert len(MEMBER_SEEDS) == MAX_MEMBERS

ABLATION_DIR = os.path.join(
    BASE,
    "checkpoints",
    "belt_ensemble_size_ablation"
)

os.makedirs(ABLATION_DIR, exist_ok=True)

RESULT_DIR = os.path.join(
    BASE,
    "results",
    "belt_ensemble_size_ablation"
)

os.makedirs(RESULT_DIR, exist_ok=True)

print("Ensemble sizes:", ENSEMBLE_SIZES)
print("Maximum members:", MAX_MEMBERS)
print("Checkpoint directory:", ABLATION_DIR)

import os, random, pickle, json, re, html, io
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFilter, ImageEnhance
from tqdm.auto import tqdm
from collections import Counter
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    matthews_corrcoef, cohen_kappa_score, confusion_matrix
)
from google.colab import drive
drive.mount('/content/drive')

NUM_CLASSES = 3
IMAGE_DIM = 1280
TEXT_DIM = 312
HIDDEN_DIM = 256
EMBED_DIM = 64
NUM_HEADS = 4
DROPOUT = 0.1
BATCH_SIZE = 32
MEMBER_EPOCHS = 100
ENSEMBLE_EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_TEXT_LENGTH = 64
BASE = '/content/drive/MyDrive/SeaBERT_Final'

IMG_ROOT = BASE
EPOCHS = 40
PATIENCE = 7
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

BATCH_SIZE = 32

Ensemble sizes: [2, 5, 10, 15, 20]
Maximum members: 20
Checkpoint directory: /content/drive/MyDrive/SeaBERT_Final/checkpoints/belt_ensemble_size_ablation
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# CELL 2: REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


print("Seed configuration ready.")

Seed configuration ready.


In [ ]:
from collections import Counter
def clean_text(text):
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'([!?.,])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_dataframe(df, text_column='description'):
    df = df.copy()
    before = len(df)
    df[text_column] = df[text_column].astype(str).apply(clean_text)
    valid = df[text_column].str.strip().str.len() > 0
    df = df[valid].reset_index(drop=True)
    print(f"Rows: {before} -> {len(df)}")
    return df

df_train = pd.read_csv(f'{BASE}/data/processed/damage_train.csv')
df_dev = pd.read_csv(f'{BASE}/data/processed/damage_dev.csv')
df_test = pd.read_csv(f'{BASE}/data/processed/damage_test.csv')

df_train = clean_dataframe(df_train)
df_dev = clean_dataframe(df_dev)
df_test = clean_dataframe(df_test)

print("\nTrain distribution:", Counter(df_train['severity']))
print("Dev distribution:", Counter(df_dev['severity']))
print("Test distribution:", Counter(df_test['severity']))

Rows: 2468 -> 2468
Rows: 529 -> 529
Rows: 529 -> 529

Train distribution: Counter({2: 1548, 1: 587, 0: 333})
Dev distribution: Counter({2: 332, 1: 126, 0: 71})
Test distribution: Counter({2: 332, 1: 126, 0: 71})


In [ ]:
# ============================================================
# CELL 3: FROZEN FEATURE EXTRACTION
# ============================================================
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from transformers import AutoTokenizer, AutoModel

TEXT_MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE).eval()
for p in text_model.parameters(): p.requires_grad = False

image_model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).features.to(DEVICE).eval()
for p in image_model.parameters(): p.requires_grad = False

image_transform = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_event(filename):
    parts = str(filename).split('/')
    return parts[1] if len(parts) >= 2 else 'unknown'

config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 62.7MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B / 62.7MB            

model.safetensors: downloading bytes:           |  0.00B            

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth




  0%|          | 0.00/20.5M [00:00<?, ?B/s]

 18%|█▊        | 3.62M/20.5M [00:00<00:00, 38.0MB/s]

 40%|███▉      | 8.12M/20.5M [00:00<00:00, 43.3MB/s]

 61%|██████    | 12.4M/20.5M [00:00<00:00, 42.6MB/s]

100%|██████████| 20.5M/20.5M [00:00<00:00, 45.4MB/s]


In [ ]:
import os
import pickle

train_cache = f'{BASE}/data/embeddings/severity_train_aug_bagging.pkl'
dev_cache   = f'{BASE}/data/embeddings/severity_dev_clean.pkl'
test_cache  = f'{BASE}/data/embeddings/severity_test_clean.pkl'

with open(train_cache, 'rb') as f:
    train_data = pickle.load(f)

with open(dev_cache, 'rb') as f:
    dev_data = pickle.load(f)

with open(test_cache, 'rb') as f:
    test_data = pickle.load(f)

print("Loaded cached features:")
print("Train:", train_data['img_maps'].shape, train_data['txt_seqs'].shape)
print("Dev:  ", dev_data['img_maps'].shape, dev_data['txt_seqs'].shape)
print("Test: ", test_data['img_maps'].shape, test_data['txt_seqs'].shape)

Loaded cached features:
Train: (3348, 1280, 7, 7) (3348, 64, 312)
Dev:   (529, 1280, 7, 7) (529, 64, 312)
Test:  (529, 1280, 7, 7) (529, 64, 312)


In [ ]:
NUM_CLASSES = 3
EVENT2IDX = {e: i for i, e in enumerate(sorted(set(train_data['events'] + dev_data['events'] + test_data['events'])))}

class SeverityDataset(Dataset):
    def __init__(self, data):
        self.images = torch.from_numpy(data['img_maps']).float()
        self.text = torch.from_numpy(data['txt_seqs']).float()
        self.masks = torch.from_numpy(data['txt_masks']).float()
        self.labels = torch.from_numpy(data['labels']).long()
        self.events = torch.tensor([EVENT2IDX[e] for e in data['events']], dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, index):
        return self.images[index], self.text[index], self.masks[index], self.labels[index], self.events[index]

train_dataset = SeverityDataset(train_data)
dev_dataset = SeverityDataset(dev_data)
test_dataset = SeverityDataset(test_data)

class_counts = np.bincount(train_data['labels'], minlength=NUM_CLASSES).astype(np.float32)
class_weights = 1.0 / np.sqrt(class_counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class counts:", class_counts, "| Class weights:", CLASS_WEIGHTS.cpu().numpy())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

Class counts: [ 900.  900. 1548.] | Class weights: [1.0859758 1.0859758 0.8280486]


In [ ]:
class CoralHead(nn.Module):
    def __init__(self, in_dim, num_classes=NUM_CLASSES):
        super().__init__()
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)
        self.bias0 = nn.Parameter(torch.zeros(1))
        self.deltas = nn.Parameter(torch.ones(max(self.num_thresholds - 1, 0)))

    def forward(self, x):
        base = self.fc(x)
        biases = [self.bias0]
        b = self.bias0
        for i in range(self.num_thresholds - 1):
            b = b - F.softplus(self.deltas[i])
            biases.append(b)
        biases = torch.cat(biases)
        return base + biases.unsqueeze(0)

class CoralLoss(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, class_weights=None):
        super().__init__()
        self.num_classes = num_classes
        self.class_weights = class_weights

    def forward(self, logits, targets):
        thresholds = torch.arange(self.num_classes - 1, device=logits.device)
        ordinal_targets = (targets.unsqueeze(1) > thresholds.unsqueeze(0)).float()
        loss = F.binary_cross_entropy_with_logits(logits, ordinal_targets, reduction='none').sum(dim=1)
        if self.class_weights is not None:
            loss = loss * self.class_weights.to(logits.device, logits.dtype)[targets]
        return loss.mean()

def decode_coral(logits, thresholds=(0.5, 0.5)):
    probabilities = torch.sigmoid(logits)
    t1, t2 = thresholds
    predictions = torch.zeros(len(logits), dtype=torch.long, device=logits.device)
    predictions[(probabilities[:, 0] > t1) & (probabilities[:, 1] <= t2)] = 1
    predictions[probabilities[:, 1] > t2] = 2
    return predictions

print("CORAL components ready. Thresholds:", NUM_CLASSES - 1)

CORAL components ready. Thresholds: 2


In [ ]:
class AdapterPoolText(nn.Module):
    def __init__(self, input_dim=TEXT_DIM, output_dim=HIDDEN_DIM, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(input_dim, bottleneck)
        self.up = nn.Linear(bottleneck, input_dim)
        self.norm = nn.LayerNorm(input_dim)
        self.projection = nn.Linear(input_dim, output_dim)

    def forward(self, text, mask):
        residual = text
        x = F.gelu(self.down(text))
        x = self.up(x)
        x = self.norm(residual + x)
        mask = mask.unsqueeze(-1)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        return self.projection(x)


class BELT(nn.Module):
    """One ensemble member: own image/text projections, own bidirectional
    cross-attention, own self-attention refinement, own 64-D embedding.
    Each of NUM_MEMBERS BELT instances is independent and trains on its
    own balanced bootstrap subset (drawn from the augmented pool)."""
    def __init__(self):
        super().__init__()
        self.image_projection = nn.Sequential(
            nn.Linear(IMAGE_DIM, HIDDEN_DIM), nn.LayerNorm(HIDDEN_DIM), nn.ReLU()
        )
        self.text_projection = AdapterPoolText()

        self.image_to_text = nn.MultiheadAttention(HIDDEN_DIM, NUM_HEADS, dropout=DROPOUT, batch_first=True)
        self.text_to_image = nn.MultiheadAttention(HIDDEN_DIM, NUM_HEADS, dropout=DROPOUT, batch_first=True)
        self.image_norm = nn.LayerNorm(HIDDEN_DIM)
        self.text_norm = nn.LayerNorm(HIDDEN_DIM)

        self.self_attention = nn.TransformerEncoderLayer(
            d_model=HIDDEN_DIM, nhead=NUM_HEADS, dim_feedforward=512,
            dropout=DROPOUT, batch_first=True, norm_first=True
        )

        self.embedding = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, EMBED_DIM), nn.LayerNorm(EMBED_DIM), nn.ReLU()
        )
        self.coral_head = CoralHead(EMBED_DIM, num_classes=NUM_CLASSES)

    def embed(self, image, text, mask):
        if image.ndim == 4:
            image = F.adaptive_avg_pool2d(image, 1).flatten(1)
        v = self.image_projection(image)
        t = self.text_projection(text, mask)

        v_tok, t_tok = v.unsqueeze(1), t.unsqueeze(1)

        v_att, _ = self.image_to_text(v_tok, t_tok, t_tok)   # V <- T
        t_att, _ = self.text_to_image(t_tok, v_tok, v_tok)   # T <- V (parallel, from original tokens)

        v_new = self.image_norm(v_tok + v_att)
        t_new = self.text_norm(t_tok + t_att)

        pair = torch.cat([v_new, t_new], dim=1)
        pair = self.self_attention(pair)
        fused = pair.flatten(start_dim=1)

        return self.embedding(fused)

    def forward(self, image, text, mask):
        return self.coral_head(self.embed(image, text, mask))


_m = BELT().to(DEVICE)
_img, _txt, _mask, _label, _event = next(iter(train_loader))
_img, _txt, _mask = _img.to(DEVICE), _txt.to(DEVICE), _mask.to(DEVICE)
with torch.no_grad():
    _emb = _m.embed(_img, _txt, _mask)
    _logits = _m(_img, _txt, _mask)
print("Embedding shape:", tuple(_emb.shape), "| CORAL logits shape:", tuple(_logits.shape))
del _m, _img, _txt, _mask, _label, _event, _emb, _logits

Embedding shape: (32, 64) | CORAL logits shape: (32, 2)


In [ ]:
# ============================================================
# CELL 3: BELT MODEL FACTORY
# ============================================================

def build_belt():
    """
    Creates a completely independent BELT model.

    IMPORTANT:
    This must return the exact final BELT architecture used
    in your main experiment.
    """

    model = BELT().to(DEVICE)

    return model


# Test construction
_test_model = build_belt()

print(
    f"BELT parameters: "
    f"{sum(p.numel() for p in _test_model.parameters()):,}"
)

print(
    f"Trainable parameters: "
    f"{sum(p.numel() for p in _test_model.parameters() if p.requires_grad):,}"
)

del _test_model

BELT parameters: 1,537,002
Trainable parameters: 1,537,002


In [ ]:
# ============================================================
# CELL 4: TRAIN ONE BELT MEMBER
# ============================================================

def train_one_belt(
    seed,
    epochs=40,
    patience_limit=7,
    lr=1e-4,
    weight_decay=1e-4
):

    print("\n" + "=" * 70)
    print(f"TRAINING BELT MEMBER | SEED = {seed}")
    print("=" * 70)

    set_seed(seed)

    model = build_belt()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=1e-6
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )

    best_f1 = -np.inf
    best_state = None
    patience = 0

    history = {
        "loss": [],
        "dev_f1": [],
        "dev_acc": []
    }

    for epoch in range(epochs):

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        model.train()

        running_loss = 0.0

        for batch in train_loader:

            img, txt, mask, label, *rest = batch

            img = img.to(DEVICE, non_blocking=True)
            txt = txt.to(DEVICE, non_blocking=True)
            mask = mask.to(DEVICE, non_blocking=True)
            label = label.to(DEVICE, non_blocking=True).long()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=(DEVICE.type == "cuda")
            ):

                logits = model(img, txt, mask)

                # ------------------------------------------------
                # BELT must return [B, 2] CORAL logits for 3 classes
                # ------------------------------------------------

                loss = coral_loss(
                    logits,
                    label
                )

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # ----------------------------------------------------
        # DEV EVALUATION
        # ----------------------------------------------------

        dev_f1, dev_acc = evaluate_single_belt(
            model,
            dev_loader
        )

        scheduler.step()

        history["loss"].append(train_loss)
        history["dev_f1"].append(dev_f1)
        history["dev_acc"].append(dev_acc)

        if dev_f1 > best_f1:

            best_f1 = dev_f1
            patience = 0

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            marker = "[BEST]"

        else:

            patience += 1
            marker = ""

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss {train_loss:.4f} | "
            f"Dev F1 {dev_f1:.4f} | "
            f"Dev Acc {dev_acc:.4f} {marker}"
        )

        if patience >= patience_limit:

            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    checkpoint_path = os.path.join(
        ABLATION_DIR,
        f"belt_member_seed_{seed}.pth"
    )

    torch.save(
        model.state_dict(),
        checkpoint_path
    )

    print(
        f"Saved member checkpoint:\n{checkpoint_path}"
    )

    return model, history

In [ ]:
# ============================================================
# CELL 5: CORAL LOSS
# ============================================================

import torch.nn.functional as F


def coral_targets(labels, num_classes=3):

    """
    Converts class labels:

        0 -> [0, 0]
        1 -> [1, 0]
        2 -> [1, 1]

    for ordinal CORAL learning.
    """

    targets = torch.zeros(
        labels.size(0),
        num_classes - 1,
        device=labels.device
    )

    for k in range(num_classes - 1):
        targets[:, k] = (labels > k).float()

    return targets


def coral_loss(logits, labels):

    targets = coral_targets(
        labels,
        num_classes=3
    )

    return F.binary_cross_entropy_with_logits(
        logits,
        targets
    )

In [ ]:
# ============================================================
# CELL 6: CORAL -> CLASS PROBABILITIES
# ============================================================

@torch.no_grad()
def coral_to_probabilities(logits):

    """
    logits shape:
        [B, 2]

    Represents:

        P(y > 0)
        P(y > 1)

    Returns:

        [P(class 0), P(class 1), P(class 2)]
    """

    cumulative = torch.sigmoid(logits)

    p_gt_0 = cumulative[:, 0]
    p_gt_1 = cumulative[:, 1]

    p0 = 1.0 - p_gt_0
    p1 = p_gt_0 - p_gt_1
    p2 = p_gt_1

    probs = torch.stack(
        [p0, p1, p2],
        dim=1
    )

    # Numerical protection
    probs = torch.clamp(
        probs,
        min=0.0,
        max=1.0
    )

    # Renormalize
    probs = probs / probs.sum(
        dim=1,
        keepdim=True
    ).clamp(min=1e-8)

    return probs

In [ ]:
# ============================================================
# CELL 7: SINGLE BELT EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_single_belt(model, loader):

    model.eval()

    targets = []
    predictions = []

    for batch in loader:

        img, txt, mask, label, *rest = batch

        img = img.to(DEVICE, non_blocking=True)
        txt = txt.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(
                img,
                txt,
                mask
            )

            probs = coral_to_probabilities(logits)

        pred = probs.argmax(dim=1)

        predictions.extend(
            pred.cpu().numpy().tolist()
        )

        targets.extend(
            label.long().cpu().numpy().tolist()
        )

    f1 = f1_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    acc = accuracy_score(
        targets,
        predictions
    )

    return f1, acc

In [ ]:
# ============================================================
# CELL 8: TRAIN 20 INDEPENDENT BELT MEMBERS
# ============================================================

member_models = []
member_histories = []

for member_idx in range(MAX_MEMBERS):

    seed = MEMBER_SEEDS[member_idx]

    checkpoint_path = os.path.join(
        ABLATION_DIR,
        f"belt_member_seed_{seed}.pth"
    )

    model = build_belt()

    if os.path.exists(checkpoint_path):

        print(
            f"\nLoading existing member "
            f"{member_idx + 1}/{MAX_MEMBERS} "
            f"(seed={seed})"
        )

        state = torch.load(
            checkpoint_path,
            map_location="cpu"
        )

        model.load_state_dict(state)

        model.to(DEVICE)

        member_models.append(model)
        member_histories.append(None)

    else:

        model, history = train_one_belt(
            seed=seed
        )

        member_models.append(model)
        member_histories.append(history)

print("\n" + "=" * 70)
print(f"READY: {len(member_models)} BELT MEMBERS")
print("=" * 70)


TRAINING BELT MEMBER | SEED = 42
Epoch 01 | Loss 0.5048 | Dev F1 0.4193 | Dev Acc 0.6635 [BEST]
Epoch 02 | Loss 0.3845 | Dev F1 0.4216 | Dev Acc 0.6673 [BEST]
Epoch 03 | Loss 0.3205 | Dev F1 0.4300 | Dev Acc 0.6616 [BEST]
Epoch 04 | Loss 0.2695 | Dev F1 0.4399 | Dev Acc 0.6749 [BEST]
Epoch 05 | Loss 0.2375 | Dev F1 0.4432 | Dev Acc 0.6730 [BEST]
Epoch 06 | Loss 0.2132 | Dev F1 0.4318 | Dev Acc 0.6541 
Epoch 07 | Loss 0.1996 | Dev F1 0.4395 | Dev Acc 0.6692 
Epoch 08 | Loss 0.1831 | Dev F1 0.4231 | Dev Acc 0.6560 
Epoch 09 | Loss 0.1750 | Dev F1 0.4595 | Dev Acc 0.6805 [BEST]
Epoch 10 | Loss 0.1684 | Dev F1 0.4357 | Dev Acc 0.6635 
Epoch 11 | Loss 0.1591 | Dev F1 0.4397 | Dev Acc 0.6673 
Epoch 12 | Loss 0.1537 | Dev F1 0.4589 | Dev Acc 0.6843 
Epoch 13 | Loss 0.1495 | Dev F1 0.4351 | Dev Acc 0.6635 
Epoch 14 | Loss 0.1457 | Dev F1 0.4289 | Dev Acc 0.6616 
Epoch 15 | Loss 0.1378 | Dev F1 0.4565 | Dev Acc 0.6843 
Epoch 16 | Loss 0.1342 | Dev F1 0.4622 | Dev Acc 0.6805 [BEST]
Epoch 17 | L

In [ ]:
# ============================================================
# CELL 9: COLLECT TEST PREDICTIONS FROM ALL BELT MEMBERS
# ============================================================

@torch.no_grad()
def collect_member_predictions(model, loader):

    model.eval()

    all_probs = []
    all_targets = []

    for batch in loader:

        img, txt, mask, label, *rest = batch

        img = img.to(DEVICE, non_blocking=True)
        txt = txt.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(
                img,
                txt,
                mask
            )

            probs = coral_to_probabilities(logits)

        all_probs.append(
            probs.cpu()
        )

        all_targets.extend(
            label.long().cpu().numpy().tolist()
        )

    all_probs = torch.cat(
        all_probs,
        dim=0
    ).numpy()

    all_targets = np.asarray(
        all_targets
    )

    return all_probs, all_targets


member_predictions = []

for i, model in enumerate(member_models):

    print(
        f"Collecting predictions: "
        f"BELT member {i+1}/{MAX_MEMBERS}"
    )

    probs, targets = collect_member_predictions(
        model,
        test_loader
    )

    member_predictions.append(probs)


member_predictions = np.stack(
    member_predictions,
    axis=0
)

print("\nPrediction tensor shape:")
print(member_predictions.shape)

print(
    "Expected shape:",
    f"({MAX_MEMBERS}, {len(targets)}, 3)"
)


Prediction tensor shape:
(20, 529, 3)
Expected shape: (20, 529, 3)


In [ ]:
# ============================================================
# CELL 10: ENSEMBLE SIZE ABLATION
# ============================================================

def evaluate_ensemble(
    member_probs,
    targets,
    n_members
):

    # --------------------------------------------------------
    # Take the FIRST n independently trained BELT members
    # --------------------------------------------------------

    selected = member_probs[:n_members]

    # --------------------------------------------------------
    # Probability averaging
    # --------------------------------------------------------

    ensemble_probs = selected.mean(
        axis=0
    )

    # --------------------------------------------------------
    # Final prediction
    # --------------------------------------------------------

    predictions = ensemble_probs.argmax(
        axis=1
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        targets,
        predictions
    )

    macro_f1 = f1_score(
        targets,
        predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        targets,
        predictions,
        average="weighted",
        zero_division=0
    )

    mcc = matthews_corrcoef(
        targets,
        predictions
    )

    kappa = cohen_kappa_score(
        targets,
        predictions
    )

    qwk = cohen_kappa_score(
        targets,
        predictions,
        weights="quadratic"
    )

    return {
        "members": n_members,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "mcc": mcc,
        "cohen_kappa": kappa,
        "quadratic_kappa": qwk,
        "predictions": predictions,
        "probabilities": ensemble_probs
    }


ablation_results = []

for n in ENSEMBLE_SIZES:

    result = evaluate_ensemble(
        member_predictions,
        targets,
        n
    )

    ablation_results.append(result)

    print(
        f"\nBELT-{n}"
    )

    print(
        f"Accuracy      : {result['accuracy']:.4f}"
    )

    print(
        f"Macro F1      : {result['macro_f1']:.4f}"
    )

    print(
        f"Weighted F1   : {result['weighted_f1']:.4f}"
    )

    print(
        f"MCC           : {result['mcc']:.4f}"
    )

    print(
        f"Cohen Kappa   : {result['cohen_kappa']:.4f}"
    )

    print(
        f"Quadratic Kappa: {result['quadratic_kappa']:.4f}"
    )


BELT-2
Accuracy      : 0.6427
Macro F1      : 0.4361
Weighted F1   : 0.5703
MCC           : 0.2857
Cohen Kappa   : 0.2562
Quadratic Kappa: 0.3990

BELT-5
Accuracy      : 0.6408
Macro F1      : 0.4186
Weighted F1   : 0.5601
MCC           : 0.2834
Cohen Kappa   : 0.2512
Quadratic Kappa: 0.4003

BELT-10
Accuracy      : 0.6446
Macro F1      : 0.4169
Weighted F1   : 0.5608
MCC           : 0.2795
Cohen Kappa   : 0.2444
Quadratic Kappa: 0.3941

BELT-15
Accuracy      : 0.6427
Macro F1      : 0.4174
Weighted F1   : 0.5599
MCC           : 0.2788
Cohen Kappa   : 0.2449
Quadratic Kappa: 0.3934

BELT-20
Accuracy      : 0.6446
Macro F1      : 0.4201
Weighted F1   : 0.5620
MCC           : 0.2859
Cohen Kappa   : 0.2518
Quadratic Kappa: 0.4036


In [ ]:
# ============================================================
# CELL: BELT MEMBER-COUNT ABLATION
# Aggregate MEMBER EMBEDDINGS -> Shared Classifier
# ============================================================

import copy
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# BELT counts to evaluate
# ------------------------------------------------------------
BELT_COUNTS = [2, 5, 10, 15, 20]

# members must already contain the trained BELT members
# Example:
# members = [belt1, belt2, ..., belt20]

assert len(members) >= max(BELT_COUNTS), (
    f"Need at least {max(BELT_COUNTS)} trained BELT members, "
    f"but only {len(members)} are available."
)

print(f"Available BELT members: {len(members)}")


# ------------------------------------------------------------
# 1. Extract the fused embedding from ONE BELT
# ------------------------------------------------------------
@torch.no_grad()
def get_belt_embedding(member, img, txt, mask):
    """
    Returns the multimodal embedding produced by one BELT
    BEFORE the final classification head.

    IMPORTANT:
    This must correspond to the same fused representation
    used by the main BELT architecture.
    """

    member.eval()

    # Use the embed method directly
    embedding = member.embed(
        img,
        txt,
        mask
    )

    return embedding


# ------------------------------------------------------------
# 2. Aggregate embeddings from N BELT members
# ------------------------------------------------------------
@torch.no_grad()
def aggregate_belt_embeddings(
    selected_members,
    loader
):
    """
    Each BELT independently produces an embedding.

    Embeddings are then averaged across BELT members:

        Z = (1/N) * sum_i Z_i

    The classifier receives this aggregated embedding.
    """

    all_embeddings = []
    all_labels = []

    for img, txt, mask, label, eid in loader:

        img = img.to(DEVICE, non_blocking=True)
        txt = txt.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)

        member_embeddings = []

        for member in selected_members:

            z_i = get_belt_embedding(
                member,
                img,
                txt,
                mask
            )

            member_embeddings.append(z_i)

        # [N, B, D]
        member_embeddings = torch.stack(
            member_embeddings,
            dim=0
        )

        # Aggregate BELT embeddings
        # [B, D]
        aggregated_embedding = member_embeddings.mean(dim=0)

        all_embeddings.append(
            aggregated_embedding.cpu()
        )

        all_labels.append(
            label.long().cpu()
        )

    return (
        torch.cat(all_embeddings, dim=0),
        torch.cat(all_labels, dim=0)
    )


# ------------------------------------------------------------
# 3. Shared classifier trained on aggregated BELT embedding
# ------------------------------------------------------------
class BELTAggregatedClassifier(nn.Module):

    def __init__(self, embedding_dim=128, num_classes=3):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)


# ------------------------------------------------------------
# 4. Train classifier for a given number of BELTs
# ------------------------------------------------------------
def train_aggregated_classifier(
    train_embeddings,
    train_labels,
    dev_embeddings,
    dev_labels,
    epochs=30,
    lr=1e-3,
    patience=6
):

    train_dataset = torch.utils.data.TensorDataset(
        train_embeddings,
        train_labels
    )

    dev_dataset = torch.utils.data.TensorDataset(
        dev_embeddings,
        dev_labels
    )

    train_loader_local = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True
    )

    dev_loader_local = DataLoader(
        dev_dataset,
        batch_size=128,
        shuffle=False
    )

    embedding_dim = train_embeddings.shape[1]

    classifier = BELTAggregatedClassifier(
        embedding_dim=embedding_dim,
        num_classes=3
    ).to(DEVICE)

    # Class-weighted CE for the severity imbalance
    counts = torch.bincount(
        train_labels,
        minlength=3
    ).float()

    weights = counts.sum() / (
        3.0 * counts.clamp(min=1)
    )

    weights = weights.to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=weights
    )

    optimizer = torch.optim.AdamW(
        classifier.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    best_f1 = -1
    best_state = None
    wait = 0

    for epoch in range(epochs):

        classifier.train()
        train_loss = 0.0

        for x, y in train_loader_local:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = classifier(x)

            loss = criterion(
                logits,
                y
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                classifier.parameters(),
                1.0
            )

            optimizer.step()

            train_loss += loss.item()

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------
        classifier.eval()

        val_targets = []
        val_preds = []

        with torch.no_grad():

            for x, y in dev_loader_local:

                x = x.to(DEVICE)
                y = y.to(DEVICE)

                logits = classifier(x)

                pred = logits.argmax(
                    dim=1
                )

                val_targets.extend(
                    y.cpu().numpy()
                )

                val_preds.extend(
                    pred.cpu().numpy()
                )

        val_f1 = f1_score(
            val_targets,
            val_preds,
            average="macro",
            zero_division=0
        )

        if val_f1 > best_f1:

            best_f1 = val_f1

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in classifier.state_dict().items()
            }

            wait = 0

        else:
            wait += 1

        if wait >= patience:
            break

    classifier.load_state_dict(
        best_state
    )

    return classifier, best_f1


# ------------------------------------------------------------
# 5. Evaluate classifier
# ------------------------------------------------------------
@torch.no_grad()
def evaluate_aggregated_classifier(
    classifier,
    embeddings,
    labels
):

    classifier.eval()

    loader = DataLoader(
        torch.utils.data.TensorDataset(
            embeddings,
            labels
        ),
        batch_size=128,
        shuffle=False
    )

    targets = []
    preds = []

    for x, y in loader:

        x = x.to(DEVICE)

        logits = classifier(x)

        pred = logits.argmax(
            dim=1
        )

        targets.extend(
            y.numpy()
        )

        preds.extend(
            pred.cpu().numpy()
        )

    targets = np.asarray(targets)
    preds = np.asarray(preds)

    return {
        "accuracy": accuracy_score(
            targets,
            preds
        ),

        "macro_f1": f1_score(
            targets,
            preds,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            targets,
            preds,
            average="weighted",
            zero_division=0
        ),

        "mcc": matthews_corrcoef(
            targets,
            preds
        ),

        "kappa": cohen_kappa_score(
            targets,
            preds
        ),

        "qwk": cohen_kappa_score(
            targets,
            preds,
            weights="quadratic"
        ),

        "cm": confusion_matrix(
            targets,
            preds
        ),

        "targets": targets,
        "preds": preds
    }


# ============================================================
# 6. Run BELT-count ablation
# ============================================================

belt_count_results = {}

for n_belts in BELT_COUNTS:

    print("\n" + "=" * 70)
    print(f"BELT MEMBER COUNT = {n_belts}")
    print("=" * 70)

    selected_members = members[:n_belts]

    # --------------------------------------------------------
    # Aggregate TRAIN embeddings
    # --------------------------------------------------------
    train_emb, train_lab = aggregate_belt_embeddings(
        selected_members,
        train_loader
    )

    # --------------------------------------------------------
    # Aggregate DEV embeddings
    # --------------------------------------------------------
    dev_emb, dev_lab = aggregate_belt_embeddings(
        selected_members,
        dev_loader
    )

    # --------------------------------------------------------
    # Aggregate TEST embeddings
    # --------------------------------------------------------
    test_emb, test_lab = aggregate_belt_embeddings(
        selected_members,
        test_loader
    )

    print(
        f"Aggregated embedding shape: "
        f"{tuple(train_emb.shape)}"
    )

    # --------------------------------------------------------
    # Train classifier AFTER aggregation
    # --------------------------------------------------------
    classifier, best_dev_f1 = train_aggregated_classifier(
        train_emb,
        train_lab,
        dev_emb,
        dev_lab
    )

    # --------------------------------------------------------
    # Test
    # --------------------------------------------------------
    result = evaluate_aggregated_classifier(
        classifier,
        test_emb,
        test_lab
    )

    result["num_belts"] = n_belts
    result["best_dev_f1"] = best_dev_f1

    belt_count_results[n_belts] = result

    print(
        f"\n{n_belts} BELTs:"
    )

    print(
        f"Accuracy     : "
        f"{result['accuracy']:.4f}"
    )

    print(
        f"Macro F1     : "
        f"{result['macro_f1']:.4f}"
    )

    print(
        f"Weighted F1  : "
        f"{result['weighted_f1']:.4f}"
    )

    print(
        f"MCC          : "
        f"{result['mcc']:.4f}"
    )

    print(
        f"Kappa        : "
        f"{result['kappa']:.4f}"
    )

    print(
        f"QWK          : "
        f"{result['qwk']:.4f}"
    )


# ============================================================
# 7. Final comparison table
# ============================================================

print("\n")
print("=" * 80)
print("BELT MEMBER COUNT ABLATION")
print("=" * 80)

print(
    f"{'BELTs':>8}"
    f"{'Accuracy':>12}"
    f"{'Macro F1':>12}"
    f"{'Weighted F1':>14}"
    f"{'MCC':>10}"
    f"{'Kappa':>10}"
    f"{'QWK':>10}"
)

print("-" * 80)

for n in BELT_COUNTS:

    r = belt_count_results[n]

    print(
        f"{n:>8}"
        f"{r['accuracy']:>12.4f}"
        f"{r['macro_f1']:>12.4f}"
        f"{r['weighted_f1']:>14.4f}"
        f"{r['mcc']:>10.4f}"
        f"{r['kappa']:>10.4f}"
        f"{r['qwk']:>10.4f}"
    )


# ============================================================
# 8. Detailed report for the best BELT count
# ============================================================

best_n = max(
    BELT_COUNTS,
    key=lambda n: belt_count_results[n]["macro_f1"]
)

best_result = belt_count_results[best_n]

print("\n" + "=" * 80)
print(
    f"BEST BELT CONFIGURATION: {best_n} MEMBERS"
)
print("=" * 80)

print(
    classification_report(
        best_result["targets"],
        best_result["preds"],
        target_names=[
            "Little/No Damage",
            "Mild Damage",
            "Severe Damage"
        ],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(best_result["cm"])

Available BELT members: 20

BELT MEMBER COUNT = 2
Aggregated embedding shape: (3348, 64)

2 BELTs:
Accuracy     : 0.6352
Macro F1     : 0.5352
Weighted F1  : 0.6393
MCC          : 0.3223
Kappa        : 0.3191
QWK          : 0.4638

BELT MEMBER COUNT = 5
Aggregated embedding shape: (3348, 64)

5 BELTs:
Accuracy     : 0.6295
Macro F1     : 0.5245
Weighted F1  : 0.6339
MCC          : 0.3140
Kappa        : 0.3109
QWK          : 0.4560

BELT MEMBER COUNT = 10
Aggregated embedding shape: (3348, 64)

10 BELTs:
Accuracy     : 0.6503
Macro F1     : 0.5284
Weighted F1  : 0.6412
MCC          : 0.3121
Kappa        : 0.3098
QWK          : 0.4332

BELT MEMBER COUNT = 15
Aggregated embedding shape: (3348, 64)

15 BELTs:
Accuracy     : 0.6616
Macro F1     : 0.5224
Weighted F1  : 0.6436
MCC          : 0.3178
Kappa        : 0.3128
QWK          : 0.4411

BELT MEMBER COUNT = 20
Aggregated embedding shape: (3348, 64)

20 BELTs:
Accuracy     : 0.6522
Macro F1     : 0.5170
Weighted F1  : 0.6325
MCC          